# Proyecto 2 - Modelo LSTM para Pronóstico del Clima

## 04_LSTM_Forecasting.ipynb

Este notebook usa PyTorch y LSTM para generar predicciones de `meantemp` en la serie de tiempo climática.

### Objetivos
- Cargar y preparar las series de tiempo
- Construir un modelo LSTM con PyTorch
- Entrenar y evaluar las predicciones
- Visualizar resultados reales vs predichos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

sns.set(style='whitegrid', font_scale=1.1)

train_path = '../data/DailyDelhiClimateTrain.csv'
test_path = '../data/DailyDelhiClimateTest.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

## Preprocesamiento y secuencias

In [ ]:
series = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)
features = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
scaler = MinMaxScaler()
scaled = scaler.fit_transform(series[features])

lookback = 7

X = []
y = []
for i in range(len(scaled) - lookback):
    X.append(scaled[i:i+lookback, :])
    y.append(scaled[i+lookback, 0])

X = np.array(X)
y = np.array(y)

train_size = len(train_df) - lookback
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

print('X_train', X_train.shape, 'X_test', X_test.shape)

## Modelo LSTM en PyTorch

In [ ]:
class ClimateLSTM(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        x = output[:, -1, :]
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = ClimateLSTM(input_size=4)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

## Entrenamiento

In [ ]:
epochs = 80
batch_size = 32

for epoch in range(1, epochs + 1):
    model.train()
    permutation = torch.randperm(X_train_t.size(0))
    epoch_loss = 0
    for i in range(0, X_train_t.size(0), batch_size):
        idx = permutation[i:i + batch_size]
        batch_X = X_train_t[idx]
        batch_y = y_train_t[idx]

        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_X.size(0)

    epoch_loss /= X_train_t.size(0)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch}/{epochs} - Loss: {epoch_loss:.6f}')

## Evaluación

In [ ]:
model.eval()
with torch.no_grad():
    preds_train = model(X_train_t).numpy().flatten()
    preds_test = model(X_test_t).numpy().flatten()

train_rmse = np.sqrt(mean_squared_error(y_train, preds_train))
val_rmse = np.sqrt(mean_squared_error(y_test, preds_test))
print('Train RMSE:', train_rmse)
print('Test RMSE:', val_rmse)

# Revertir escala para la variable de temperatura
inv_test = np.zeros((len(preds_test), len(features)))
inv_test[:, 0] = preds_test
inv_test = scaler.inverse_transform(inv_test)[:, 0]

inv_real = np.zeros((len(y_test), len(features)))
inv_real[:, 0] = y_test
inv_real = scaler.inverse_transform(inv_real)[:, 0]

plt.figure(figsize=(14, 6))
plt.plot(inv_real, label='Real')
plt.plot(inv_test, label='Predicho')
plt.title('Predicción de temperatura media (meantemp)')
plt.xlabel('Muestra')
plt.ylabel('meantemp')
plt.legend()
plt.show()

## Conclusión

El modelo LSTM ofrece una primera aproximación de pronóstico sobre la temperatura. Se recomienda continuar iterando con más características, regularización y validación cruzada temporal para mejorar la precisión.